# 🚀 NVIDIA RAPIDS cuDF Benchmark — Bengaluru Commute Decision Tool

**Objective:** Demonstrate GPU-accelerated data processing using `cudf.pandas` on the Bengaluru traffic cleaning pipeline.

**Setup:**
- Runtime: Google Colab with **T4 GPU**
- Dataset: Scaled to ~1M rows
- Comparison: Standard pandas (CPU) vs cudf.pandas (GPU)

---

## ⚙️ Prerequisites

1. Go to `Runtime → Change runtime type → T4 GPU`
2. Upload `Banglore_traffic_Dataset.csv` to the Colab session

In [ ]:
# Step 0: Verify GPU
!nvidia-smi

In [ ]:
# Step 1: Install RAPIDS cuDF (Colab-compatible)
!pip install --extra-index-url=https://pypi.nvidia.com cudf-cu12 --quiet

In [ ]:
# Step 2: Upload your dataset
from google.colab import files
import os

if not os.path.exists('Banglore_traffic_Dataset.csv'):
    print("Please upload Banglore_traffic_Dataset.csv")
    uploaded = files.upload()
else:
    print("Dataset already present.")

---

## 📊 Data Scaling to ~1M Rows

The original dataset has ~20K rows. We realistically upscale it to **~1,000,000 rows** by resampling with noise injection to simulate a large-scale production dataset.

In [ ]:
import pandas as pd
import numpy as np
import time

# Load original dataset
df_orig = pd.read_csv('Banglore_traffic_Dataset.csv')
print(f"Original dataset: {len(df_orig):,} rows, {len(df_orig.columns)} columns")
print(f"Columns: {list(df_orig.columns)}")

# Scale to ~1M rows with realistic noise
TARGET_ROWS = 1_000_000
repeats = TARGET_ROWS // len(df_orig) + 1

rng = np.random.default_rng(42)
df_scaled = pd.concat([df_orig] * repeats, ignore_index=True).head(TARGET_ROWS).copy()

# Add realistic noise to numeric columns to prevent trivial caching
numeric_cols = df_scaled.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    noise = rng.normal(0, df_scaled[col].std() * 0.05, size=len(df_scaled))
    df_scaled[col] = df_scaled[col] + noise

# Inject ~2% nulls for realistic cleaning workload
for col in numeric_cols:
    null_mask = rng.random(len(df_scaled)) < 0.02
    df_scaled.loc[null_mask, col] = np.nan

# Save scaled dataset
df_scaled.to_csv('scaled_traffic_1M.csv', index=False)
print(f"\nScaled dataset: {len(df_scaled):,} rows")
print(f"Null counts:\n{df_scaled.isnull().sum()[df_scaled.isnull().sum() > 0]}")

---

## 🧹 Define the Cleaning Pipeline

This is the **exact same** cleaning logic from `scripts/clean_data.py`, packaged as a single function. No code changes are needed for cuDF — the `cudf.pandas` accelerator handles it transparently.

In [ ]:
def run_cleaning_pipeline(csv_path):
    """
    Full cleaning pipeline — identical logic for CPU and GPU.
    This is the core ETL from clean_data.py.
    """
    import pandas as pd
    import numpy as np

    # 1. Load
    df = pd.read_csv(csv_path)

    # 2. Normalise column names
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(r'[\\s/&]+', '_', regex=True)
          .str.replace(r'[^a-z0-9_]', '', regex=True)
    )

    # 3. Parse date
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

    # 4. Drop fully-empty rows
    df.dropna(how='all', inplace=True)

    # 5. Drop duplicates
    df = df[~df.duplicated()]

    # 6. Fill numeric nulls with median
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    # 7. Fill categorical nulls with mode
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if df[col].isnull().any():
            mode_val = df[col].mode(dropna=True)
            if not mode_val.empty:
                df[col] = df[col].fillna(mode_val.iloc[0])

    # 8. Rename columns
    rename_map = {
        'roadintersection_name': 'route',
        'road_intersection_name': 'route',
        'area_name': 'area',
    }
    for old, new in rename_map.items():
        if old in df.columns:
            df.rename(columns={old: new}, inplace=True)

    # 9. Congestion column normalisation
    cong_col = next((c for c in df.columns if 'congestion' in c and 'level' in c), None)
    if cong_col and cong_col != 'congestion_level':
        df.rename(columns={cong_col: 'congestion_level'}, inplace=True)

    # 10. Risk score computation (6-component weighted)
    def minmax(s, lo, hi):
        return ((s.clip(lo, hi) - lo) / (hi - lo)).fillna(0.0)

    cong = minmax(df.get('congestion_level', pd.Series(50, index=df.index)), 0, 100)
    cap = minmax(df.get('road_capacity_utilization', pd.Series(50, index=df.index)), 0, 100)

    tti_col = next((c for c in df.columns if 'travel_time' in c), None)
    tti = minmax(df[tti_col], 1.0, 1.5) if tti_col else pd.Series(0.5, index=df.index)

    rain = pd.Series(0.0, index=df.index)  # no weather merge in benchmark
    vis = pd.Series(0.5, index=df.index)

    inc_col = next((c for c in df.columns if 'incident' in c), None)
    inc = minmax(df[inc_col], 0, 7) if inc_col else pd.Series(0.0, index=df.index)

    risk = (cong * 0.40 + cap * 0.20 + tti * 0.10 +
            rain * 0.15 + vis * 0.10 + inc * 0.05)
    df['risk_score'] = (risk * 100).clip(0, 100).round(2)

    bins = [0, 25, 50, 75, 100]
    labels = ['Low', 'Moderate', 'High', 'Critical']
    df['risk_tier'] = pd.cut(df['risk_score'], bins=bins, labels=labels, include_lowest=True)

    # 11. Time slot assignment
    slots = ['Morning Peak (07-09)', 'Midday (11-13)', 'Evening Peak (17-19)', 'Night (21-23)']
    df['time_slot'] = np.resize(slots, len(df))

    # 12. Aggregations (simulate dashboard queries)
    slot_hour = {'Morning Peak (07-09)': 8, 'Midday (11-13)': 12,
                 'Evening Peak (17-19)': 18, 'Night (21-23)': 22}
    df['hour'] = df['time_slot'].map(slot_hour)

    route_col = 'route' if 'route' in df.columns else df.columns[0]
    q1 = df.groupby(['hour', route_col]).agg(
        avg_risk=('risk_score', 'mean'),
        avg_speed=('average_speed', 'mean') if 'average_speed' in df.columns else ('risk_score', 'count'),
    ).reset_index()

    return df, q1

---

## 🐢 Benchmark 1: Standard pandas (CPU)

In [ ]:
print("="*60)
print("  BENCHMARK: Standard pandas (CPU)")
print("="*60)

t_start = time.perf_counter()
df_cpu, q1_cpu = run_cleaning_pipeline('scaled_traffic_1M.csv')
cpu_time = time.perf_counter() - t_start

print(f"\n  Rows processed : {len(df_cpu):,}")
print(f"  Output columns : {len(df_cpu.columns)}")
print(f"  Risk tiers     : {df_cpu['risk_tier'].value_counts().to_dict()}")
print(f"  ⏱️  CPU Time    : {cpu_time:.2f} seconds")
print("="*60)

---

## 🚀 Benchmark 2: cudf.pandas (GPU-Accelerated)

**Zero code changes required!** The `cudf.pandas` module intercepts pandas calls and routes them to GPU kernels automatically.

In [ ]:
# Activate cudf.pandas accelerator
%load_ext cudf.pandas

print("✅ cudf.pandas accelerator loaded!")
print("All pandas operations will now run on GPU automatically.")

In [ ]:
print("="*60)
print("  BENCHMARK: cudf.pandas (GPU-Accelerated)")
print("="*60)

t_start = time.perf_counter()
df_gpu, q1_gpu = run_cleaning_pipeline('scaled_traffic_1M.csv')
gpu_time = time.perf_counter() - t_start

print(f"\n  Rows processed : {len(df_gpu):,}")
print(f"  Output columns : {len(df_gpu.columns)}")
print(f"  Risk tiers     : {df_gpu['risk_tier'].value_counts().to_dict()}")
print(f"  ⏱️  GPU Time    : {gpu_time:.2f} seconds")
print("="*60)

---

## 📊 Results Comparison

In [ ]:
speedup = cpu_time / gpu_time

print()
print("╔" + "═"*50 + "╗")
print("║" + "  cuDF.pandas Benchmark Results".center(50) + "║")
print("╠" + "═"*50 + "╣")
print(f"║  Dataset Size    :  {len(df_cpu):>10,} rows".ljust(51) + "║")
print(f"║  CPU (pandas)    :  {cpu_time:>10.2f} seconds".ljust(51) + "║")
print(f"║  GPU (cudf)      :  {gpu_time:>10.2f} seconds".ljust(51) + "║")
print(f"║  🚀 Speedup      :  {speedup:>10.1f}x faster".ljust(51) + "║")
print("╚" + "═"*50 + "╝")
print()
print(f"✅ NVIDIA RAPIDS cudf.pandas achieved {speedup:.1f}x acceleration")
print(f"   on the Bengaluru Commute cleaning pipeline ({len(df_cpu):,} rows).")
print(f"   Zero code changes were required — same pandas API throughout.")

In [ ]:
# Visualise the speedup
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: CPU vs GPU time
bars = ax1.bar(['pandas (CPU)', 'cudf.pandas (GPU)'], [cpu_time, gpu_time],
               color=['#FF6B6B', '#4ECB71'], edgecolor='white', linewidth=2)
ax1.set_ylabel('Time (seconds)', fontsize=12, fontweight='bold')
ax1.set_title(f'Processing Time — {len(df_cpu):,} Rows', fontsize=14, fontweight='bold')
for bar, val in zip(bars, [cpu_time, gpu_time]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}s', ha='center', fontsize=13, fontweight='bold')
ax1.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#0f0f23')
ax1.tick_params(colors='white')
ax1.yaxis.label.set_color('white')
ax1.title.set_color('white')
ax1.spines['bottom'].set_color('white')
ax1.spines['left'].set_color('white')

# Speedup gauge
ax2.barh(['Speedup'], [speedup], color='#FFD93D', height=0.4, edgecolor='white')
ax2.set_xlabel('Factor', fontsize=12, fontweight='bold')
ax2.set_title('GPU Acceleration Factor', fontsize=14, fontweight='bold')
ax2.text(speedup + 0.2, 0, f'{speedup:.1f}x', va='center', fontsize=18,
         fontweight='bold', color='#FFD93D')
ax2.set_xlim(0, speedup * 1.5)
ax2.set_facecolor('#1a1a2e')
ax2.tick_params(colors='white')
ax2.xaxis.label.set_color('white')
ax2.title.set_color('white')
ax2.spines['bottom'].set_color('white')
ax2.spines['left'].set_color('white')

plt.tight_layout()
plt.savefig('cudf_benchmark_results.png', dpi=150, bbox_inches='tight',
            facecolor='#0f0f23', edgecolor='none')
plt.show()

print("\n📸 Screenshot saved as 'cudf_benchmark_results.png'")

---

## ✅ Summary

| Metric | Value |
|--------|-------|
| Dataset | ~1M rows (scaled from 20K) |
| Pipeline | Full ETL: cleaning + risk scoring + aggregation |
| CPU Runtime | See above |
| GPU Runtime | See above |
| Speedup | See above |
| Code Changes | **Zero** — `cudf.pandas` is a drop-in accelerator |
| GPU | NVIDIA T4 (Google Colab) |

**Key Takeaway:** NVIDIA RAPIDS `cudf.pandas` provides significant acceleration on tabular ETL workloads with absolutely **no code changes** required. The same pandas code runs transparently on GPU.